# Python Iterators & Generators — Lazy Evaluation & Custom Data Traversal

> **Topic:** Python Iterators & Generators | **Folder:** Data Structures & Algorithms

In Python, an **Iterable** is an object capable of returning its members one at a time,
while an **Iterator** is an object representing a stream of data that produces values on demand using `next()`.

Understanding the **Iterator Protocol** (`__iter__` and `__next__`) and **Generators** (`yield`)
is fundamental to building **memory-efficient algorithms** ($O(1)$ space) that process massive or infinite data streams.

---

## Table of Contents
1. [Iterables vs. Iterators (The Iterator Protocol)](#1.-Iterables-vs.-Iterators-(The-Iterator-Protocol))
2. [Under the Hood: How `for` Loops Work](#2.-Under-the-Hood:-How-`for`-Loops-Work)
3. [Building Custom Iterators from Scratch](#3.-Building-Custom-Iterators-from-Scratch)
4. [Custom Data Structure Traversal (LinkedList & Tree Iterators)](#4.-Custom-Data-Structure-Traversal-(LinkedList-&-Tree-Iterators))
5. [Generators & The `yield` Keyword](#5.-Generators-&-The-`yield`-Keyword)
6. [Generator Expressions vs. List Comprehensions](#6.-Generator-Expressions-vs.-List-Comprehensions)
7. [Advanced Generator Methods (`send()`, `throw()`, `close()`)](#7.-Advanced-Generator-Methods-(send(),-throw(),-close()))
8. [The `itertools` Module Essentials](#8.-The-`itertools`-Module-Essentials)
9. [Memory & Performance Benchmarking ($O(1)$ vs $O(n)$ Space)](#9.-Memory-&-Performance-Benchmarking-($O(1)$-vs-$O(n)$-Space))
10. [Quick Reference Card](#10.-Quick-Reference-Card)


---
## 1. Iterables vs. Iterators (The Iterator Protocol)

| Term | Definition | Methods Implemented |
|------|------------|---------------------|
| **Iterable** | An object that can be looped over (e.g., `list`, `tuple`, `str`, `dict`, `set`) | `__iter__()` |
| **Iterator** | A stateful stream producing one value at a time via `next()` | `__iter__()` and `__next__()` |

- Calling `iter(iterable)` returns an **iterator**.
- Calling `next(iterator)` returns the next item. When exhausted, it raises `StopIteration`.


In [ ]:
# Converting an Iterable (list) into an Iterator
numbers = [10, 20, 30]

num_iterator = iter(numbers)
print(f"Type of numbers     : {type(numbers).__name__}")
print(f"Type of num_iterator: {type(num_iterator).__name__}")

# Fetching items manually using next()
print("next() call 1:", next(num_iterator))
print("next() call 2:", next(num_iterator))
print("next() call 3:", next(num_iterator))

# 4th call raises StopIteration exception
try:
    next(num_iterator)
except StopIteration:
    print("StopIteration exception caught: Iterator is exhausted!")


---
## 2. Under the Hood: How `for` Loops Work

A Python `for` loop is syntactic sugar for obtaining an iterator with `iter()`
and repeatedly calling `next()` inside a `try...except StopIteration` block.


In [ ]:
# Simulating a Python 'for' loop manually
data = ["apple", "banana", "cherry"]

print("--- Manual 'for' loop simulation ---")
iterator = iter(data)
while True:
    try:
        item = next(iterator)
        print(f"Processing: {item}")
    except StopIteration:
        break


---
## 3. Building Custom Iterators from Scratch

To make a custom class an **iterator**, implement:
1. `__iter__(self)`: returns `self`.
2. `__next__(self)`: returns the next value, or raises `StopIteration` when complete.


In [ ]:
# Custom Range Iterator (re-implementing range() behavior)
class CustomRange:
    def __init__(self, start, stop, step=1):
        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if (self.step > 0 and self.current >= self.stop) or \
           (self.step < 0 and self.current <= self.stop):
            raise StopIteration
        value = self.current
        self.current += self.step
        return value

# Testing CustomRange
print("CustomRange(2, 10, 2):", list(CustomRange(2, 10, 2)))
print("CustomRange(10, 0, -2):", list(CustomRange(10, 0, -2)))


---
## 4. Custom Data Structure Traversal (LinkedList & Tree Iterators)

Custom iterators allow clean iteration over non-linear or linked data structures
without exposing internal node references.


In [ ]:
# 1. Singly Linked List with Iterator
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, val):
        if not self.head:
            self.head = Node(val)
            return
        curr = self.head
        while curr.next:
            curr = curr.next
        curr.next = Node(val)

    def __iter__(self):
        curr = self.head
        while curr:
            yield curr.val
            curr = curr.next

# Usage
ll = LinkedList()
for x in [10, 20, 30, 40]: ll.append(x)

print("Iterating over Linked List:", [val for val in ll])


---
## 5. Generators & The `yield` Keyword

Generators are a simple and powerful tool for creating iterators.
A generator function uses the `yield` statement instead of `return`.
When called, it returns a **generator object** that pauses execution state after every `yield`.


In [ ]:
# Fibonacci Generator (Infinite sequence)
def fibonacci_gen():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Fetch first 10 Fibonacci numbers
fib = fibonacci_gen()
first_10_fib = [next(fib) for _ in range(10)]
print("First 10 Fibonacci numbers:", first_10_fib)


---
## 6. Generator Expressions vs. List Comprehensions

Generator expressions use parentheses `(...)` instead of brackets `[...]`.
They yield values lazily, reserving $O(1)$ memory compared to $O(n)$ for list comprehensions.


In [ ]:
import sys

N = 100_000
list_comp = [x ** 2 for x in range(N)]
gen_expr  = (x ** 2 for x in range(N))

print(f"List Comprehension memory : {sys.getsizeof(list_comp):>10,} bytes")
print(f"Generator Expression memory: {sys.getsizeof(gen_expr):>10,} bytes")


---
## 7. Advanced Generator Methods (`send()`, `throw()`, `close()`)

Generators can act as **coroutines** by receiving values dynamically via `.send()`.


In [ ]:
# Generator acting as an accumulator with .send()
def running_accumulator():
    total = 0
    while True:
        val = yield total   # Receives value sent via .send()
        if val is None: break
        total += val

acc = running_accumulator()
next(acc)            # Prime the generator
print("Added 10 -> Running total:", acc.send(10))
print("Added 25 -> Running total:", acc.send(25))
print("Added 15 -> Running total:", acc.send(15))
acc.close()


---
## 8. The `itertools` Module Essentials

The standard library `itertools` module offers optimized iterator building blocks:

| Category | Functions |
|----------|-----------|
| **Infinite** | `count()`, `cycle()`, `repeat()` |
| **Terminating** | `chain()`, `islice()`, `takewhile()`, `dropwhile()`, `groupby()` |
| **Combinatoric** | `product()`, `permutations()`, `combinations()` |


In [ ]:
import itertools as it

# 1. Infinite counter + islice
evens_stream = it.count(start=0, step=2)
first_5_evens = list(it.islice(evens_stream, 5))
print("First 5 evens from count():", first_5_evens)

# 2. Chain multiple iterables
combined = list(it.chain([1, 2], ["a", "b"], [10.5]))
print("Chained iterables:", combined)

# 3. Combinations and Permutations
letters = ['A', 'B', 'C']
print("Permutations (2):", list(it.permutations(letters, 2)))
print("Combinations (2):", list(it.combinations(letters, 2)))


---
## 9. Memory & Performance Benchmarking ($O(1)$ vs $O(n)$ Space)

Processing huge data files or large mathematical sequences with iterators avoids Out-Of-Memory (OOM) errors.


In [ ]:
import timeit

# Summing squares up to 10,000,000
t_list = timeit.timeit(lambda: sum([x**2 for x in range(10_000_000)]), number=1)
t_gen  = timeit.timeit(lambda: sum(x**2 for x in range(10_000_000)), number=1)

print(f"Eager list sum time: {t_list:.4f} seconds")
print(f"Lazy gen sum time  : {t_gen:.4f} seconds")


---
## 10. Quick Reference Card


In [ ]:
# ==================================================================
# PYTHON ITERATORS & GENERATORS – QUICK REFERENCE
# ==================================================================

# --- Protocol ---
it = iter([1, 2])
print(next(it), next(it))

# --- Generator Function ---
def count_up(n):
    for i in range(1, n + 1): yield i

print(list(count_up(3)))

# --- Generator Expression ---
squares = (x**2 for x in range(5))
print(next(squares), next(squares))


---
## Summary

| Concept | Implementation | Key Advantage |
|---------|----------------|---------------|
| **Iterator Protocol** | `__iter__()` & `__next__()` | Enables custom iteration for user classes |
| **Generators** | `yield` keyword | Simplifies iterator creation; preserves state automatically |
| **Generator Expr** | `(x for x in it)` | Minimal memory overhead $O(1)$ space |
| **`itertools`** | Standard library module | Optimized, fast C-implemented iteration primitives |

---
*Next up: **Object Oriented Programming (Classes & Inheritance)***
